In [1]:

import os
import sys
import json
import pickle
import warnings
import traceback
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend, safe for headless execution
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)
 
 
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
 
# ------------------------------------------------------------------------------
# CONFIGURATION 
# ------------------------------------------------------------------------------
import os
from pathlib import Path
# Dynamically find the project root by looking for pyproject.toml
_curr = Path.cwd()
while not (_curr / 'pyproject.toml').exists() and _curr != _curr.parent:
    _curr = _curr.parent
DATASET_PATH = str(_curr / 'data' / 'dayr4.2.csv')
OUTPUT_DIR = "outputs"                 # all artifacts are written here
RANDOM_STATE = 42
TEST_SIZE = 0.20
 
# Sub-folders inside OUTPUT_DIR
DIRS = {
    "root": OUTPUT_DIR,
    "plots": os.path.join(OUTPUT_DIR, "plots"),
    "data": os.path.join(OUTPUT_DIR, "data"),
    "models": os.path.join(OUTPUT_DIR, "models"),
    "metrics": os.path.join(OUTPUT_DIR, "metrics"),
}
 
# Keywords used to automatically guess the label / target column
LABEL_KEYWORD_CANDIDATES = [
    "insider", "malicious", "label", "target", "class",
    "ground_truth", "groundtruth", "is_malicious", "is_insider",
    "threat", "anomaly", "y",
]
 
# Keywords used to automatically detect identifier columns that must not
# be used as ML features (they carry no predictive signal, only identity)
ID_KEYWORD_CANDIDATES = ["user", "id", "employee", "day", "date", "week", "session"]
 
 
def make_output_dirs():
    """Create the output directory structure if it does not already exist."""
    for path in DIRS.values():
        os.makedirs(path, exist_ok=True)
    print(f"[OK] Output directories ready under '{OUTPUT_DIR}/'")
 
 
def savefig(fig, name):
    """Save a matplotlib figure into the plots directory and close it."""
    path = os.path.join(DIRS["plots"], name)
    fig.savefig(path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"    -> saved plot: {path}")


In [2]:
# ==============================================================================
# STEP 1 (cont.) - LOAD DATASET
# ==============================================================================
 
def load_dataset(path):
    """
    Load the CERT day-level feature CSV into a pandas DataFrame.
 
    Parameters
    ----------
    path : str
        Path to dayr4.2.csv
 
    Returns
    -------
    pd.DataFrame
    """
    print("\n" + "=" * 80)
    print("STEP 1: LOAD DATASET")
    print("=" * 80)
 
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Dataset not found at '{path}'. "
        )
 
    try:
        df = pd.read_csv(path)
    except Exception as exc:
        raise RuntimeError(f"Failed to read CSV file '{path}': {exc}")
 
    print(f"[OK] Loaded '{path}' successfully.")
    return df
 
 

In [3]:
# ==============================================================================
# STEP 2 - INITIAL DATASET INSPECTION
# ==============================================================================
 
def inspect_dataset(df):
    """Print a thorough initial inspection of the raw dataset."""
    print("\n" + "=" * 80)
    print("STEP 2: INITIAL DATASET INSPECTION")
    print("=" * 80)
 
    print(f"\nShape                : {df.shape}")
    print(f"Number of rows       : {df.shape[0]}")
    print(f"Number of columns    : {df.shape[1]}")
 
    print("\nColumn names:")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i:3d}. {col}")
 
    print("\nData types:")
    print(df.dtypes)
 
    print("\nFirst 5 rows:")
    print(df.head())
 
    print("\nLast 5 rows:")
    print(df.tail())
 
    mem_bytes = df.memory_usage(deep=True).sum()
    print(f"\nMemory usage: {mem_bytes / (1024 ** 2):.2f} MB")
 

In [4]:
# ==============================================================================
# STEP 3 - MISSING VALUE ANALYSIS
# ==============================================================================
 
def missing_value_analysis(df):
    """Analyze, report, and visualize missing values in the dataset."""
    print("\n" + "=" * 80)
    print("STEP 3: MISSING VALUE ANALYSIS")
    print("=" * 80)
 
    missing_count = df.isnull().sum()
    missing_pct = (missing_count / len(df)) * 100
 
    missing_df = pd.DataFrame(
        {"missing_count": missing_count, "missing_pct": missing_pct}
    )
    missing_df = missing_df[missing_df["missing_count"] > 0].sort_values(
        "missing_count", ascending=False
    )
 
    if missing_df.empty:
        print("[OK] No missing values found in the dataset.")
    else:
        print("\nColumns with missing values:")
        print(missing_df)
 
        # Heatmap only makes sense with actual missing values present
        fig, ax = plt.subplots(figsize=(12, 6))
        sns.heatmap(df[missing_df.index].isnull(), cbar=False, cmap="viridis", ax=ax)
        ax.set_title("Missing Value Heatmap")
        savefig(fig, "missing_value_heatmap.png")
 
    return missing_df

 

In [5]:
#  ==============================================================================
# STEP 4 - DUPLICATE ANALYSIS
# ==============================================================================
 
def duplicate_analysis(df):
    """Detect and remove duplicate rows, reporting before/after sizes."""
    print("\n" + "=" * 80)
    print("STEP 4: DUPLICATE ANALYSIS")
    print("=" * 80)
 
    n_before = len(df)
    n_dupes = df.duplicated().sum()
    print(f"Rows before removal : {n_before}")
    print(f"Duplicate rows found: {n_dupes}")
 
    if n_dupes > 0:
        df = df.drop_duplicates().reset_index(drop=True)
        print(f"[OK] Duplicates removed.")
    else:
        print("[OK] No duplicate rows found.")
 
    n_after = len(df)
    print(f"Rows after removal  : {n_after}")
 
    return df
 

In [6]:
# ==============================================================================
# STEP 5 - EXPLORATORY DATA ANALYSIS
# ==============================================================================
 
def exploratory_data_analysis(df, max_plot_cols=20):
    """
    Run a professional EDA pass: overview, statistical summary,
    distribution plots, correlation analysis, constant / highly
    correlated feature detection.
 
    max_plot_cols caps the number of individual histogram/boxplot
    figures produced, to keep runtime and output size sane on datasets
    with hundreds of engineered features (typical for CERT day-level data).
    """
    print("\n" + "=" * 80)
    print("STEP 5: EXPLORATORY DATA ANALYSIS (EDA)")
    print("=" * 80)
 
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
 
    print("\n--- Dataset Overview ---")
    print(f"Total observations : {len(df)}")
    print(f"Total features      : {df.shape[1]}")
    print(f"Numerical columns   : {len(numeric_cols)}")
    print(f"Categorical columns : {len(categorical_cols)}")
 
    print("\n--- Statistical Summary (numeric columns) ---")
    if numeric_cols:
        summary = df[numeric_cols].describe().T
        summary["median"] = df[numeric_cols].median()
        print(summary)
    else:
        print("No numeric columns found.")
 
    # --------------------------------------------------------------------
    # Histograms & boxplots for numeric columns (capped for readability)
    # --------------------------------------------------------------------
    plot_numeric_cols = numeric_cols[:max_plot_cols]
    if plot_numeric_cols:
        print(f"\nGenerating histograms/boxplots for {len(plot_numeric_cols)} "
              f"numeric columns (capped at {max_plot_cols})...")
 
        n_cols_grid = 4
        n_rows_grid = int(np.ceil(len(plot_numeric_cols) / n_cols_grid))
 
        fig, axes = plt.subplots(n_rows_grid, n_cols_grid,
                                  figsize=(5 * n_cols_grid, 4 * n_rows_grid))
        axes = np.array(axes).reshape(-1)
        for i, col in enumerate(plot_numeric_cols):
            axes[i].hist(df[col].dropna(), bins=30, color="steelblue", edgecolor="black")
            axes[i].set_title(col, fontsize=9)
        for j in range(len(plot_numeric_cols), len(axes)):
            axes[j].axis("off")
        fig.suptitle("Histograms of Numeric Features", fontsize=14)
        savefig(fig, "histograms_numeric_features.png")
 
        fig, axes = plt.subplots(n_rows_grid, n_cols_grid,
                                  figsize=(5 * n_cols_grid, 4 * n_rows_grid))
        axes = np.array(axes).reshape(-1)
        for i, col in enumerate(plot_numeric_cols):
            axes[i].boxplot(df[col].dropna())
            axes[i].set_title(col, fontsize=9)
        for j in range(len(plot_numeric_cols), len(axes)):
            axes[j].axis("off")
        fig.suptitle("Boxplots of Numeric Features", fontsize=14)
        savefig(fig, "boxplots_numeric_features.png")
 
    # --------------------------------------------------------------------
    # Categorical columns: value counts & bar charts
    # --------------------------------------------------------------------
    if categorical_cols:
        print(f"\nGenerating bar charts for {len(categorical_cols)} categorical columns...")
        for col in categorical_cols:
            vc = df[col].value_counts()
            print(f"\nValue counts for '{col}':")
            print(vc.head(20))
 
            fig, ax = plt.subplots(figsize=(8, 4))
            vc.head(20).plot(kind="bar", ax=ax, color="darkorange")
            ax.set_title(f"Value Counts: {col}")
            ax.set_ylabel("Count")
            savefig(fig, f"barchart_{col}.png")
    else:
        print("\nNo categorical columns found.")
 
    # --------------------------------------------------------------------
    # Correlation matrix / heatmap
    # --------------------------------------------------------------------
    corr_matrix = pd.DataFrame()
    if len(numeric_cols) >= 2:
        print("\nComputing correlation matrix...")
        corr_matrix = df[numeric_cols].corr()
 
        fig, ax = plt.subplots(figsize=(min(0.4 * len(numeric_cols) + 4, 30),
                                         min(0.4 * len(numeric_cols) + 4, 30)))
        sns.heatmap(corr_matrix, cmap="coolwarm", center=0, ax=ax,
                    xticklabels=len(numeric_cols) <= 40,
                    yticklabels=len(numeric_cols) <= 40)
        ax.set_title("Correlation Heatmap")
        savefig(fig, "correlation_heatmap.png")
    else:
        print("\nNot enough numeric columns for a correlation matrix.")
 
    # --------------------------------------------------------------------
    # Constant columns
    # --------------------------------------------------------------------
    constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print(f"\nConstant columns detected ({len(constant_cols)}): {constant_cols}")
 
    # --------------------------------------------------------------------
    # Highly correlated feature pairs (|corr| > 0.9)
    # --------------------------------------------------------------------
    high_corr_pairs = []
    if not corr_matrix.empty:
        for i in range(len(corr_matrix.columns)):
            for j in range(i + 1, len(corr_matrix.columns)):
                val = corr_matrix.iloc[i, j]
                if pd.notnull(val) and abs(val) > 0.9:
                    high_corr_pairs.append(
                        (corr_matrix.columns[i], corr_matrix.columns[j], round(val, 3))
                    )
        print(f"\nHighly correlated feature pairs (|r| > 0.9): {len(high_corr_pairs)}")
        for a, b, v in high_corr_pairs[:30]:
            print(f"  {a}  <->  {b}   r = {v}")
 
    return {
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "constant_cols": constant_cols,
        "high_corr_pairs": high_corr_pairs,
    }
 

In [7]:
# ==============================================================================
# STEP 6 - AUTOMATIC LABEL / TARGET IDENTIFICATION
# ==============================================================================
 
def identify_label_column(df):
    """
    Identify the target/label column for this project.
 
    The target variable for this project is fixed: 'insider'
    (the ground-truth insider-threat flag produced by the lcd-dal
    feature extraction pipeline). This function no longer guesses the
    label from keyword heuristics — it validates that the expected
    column exists and is usable, and fails loudly with a clear message
    if it does not.
 
    Returns
    -------
    str : the label column name ('insider')
    """
    print("\n" + "=" * 80)
    print("STEP 6: LABEL IDENTIFICATION")
    print("=" * 80)
 
    target_col = 'insider'
 
    if target_col not in df.columns:
        # Case-insensitive fallback lookup, in case the CSV header casing
        # differs (e.g. 'Insider', 'INSIDER').
        match = [c for c in df.columns if c.lower() == target_col]
        if match:
            target_col = match[0]
            print(f"[WARN] Column case differs from expected; using '{target_col}'.")
        else:
            raise ValueError(
                f"Expected target column 'insider' was not found in the dataset. "
                f"Available columns are: {df.columns.tolist()}"
            )
 
    n_unique = df[target_col].nunique(dropna=True)
    if n_unique != 2:
        print(
            f"[WARN] Target column '{target_col}' has {n_unique} unique values "
            f"(expected 2 for a binary insider/normal label). Proceeding anyway, "
            f"but downstream metrics assume a binary target."
        )
 
    print(f"\nTarget column identified : '{target_col}'")
 
    value_counts = df[target_col].value_counts()
    total = len(df)
    print("\nClass distribution:")
    for val, cnt in value_counts.items():
        print(f"  Class {val}: {cnt} samples ({cnt / total * 100:.2f}%)")
 
    if n_unique == 2:
        minority_class = value_counts.idxmin()
        majority_class = value_counts.idxmax()
        print(f"\nAssumed malicious (minority) class : {minority_class} "
              f"({value_counts[minority_class]} samples, "
              f"{value_counts[minority_class] / total * 100:.2f}%)")
        print(f"Assumed normal (majority) class     : {majority_class} "
              f"({value_counts[majority_class]} samples, "
              f"{value_counts[majority_class] / total * 100:.2f}%)")
 
    return target_col

In [8]:
# ==============================================================================
# STEP 7 - CLASS IMBALANCE ANALYSIS
# ==============================================================================
 
def class_imbalance_analysis(df, label_col):
    """Visualize and quantify class imbalance for the target column."""
    print("\n" + "=" * 80)
    print("STEP 7: CLASS IMBALANCE ANALYSIS")
    print("=" * 80)
 
    counts = df[label_col].value_counts()
    print("\nClass distribution:")
    print(counts)
 
    imbalance_ratio = counts.max() / counts.min()
    print(f"\nImbalance ratio (majority:minority) = {imbalance_ratio:.2f} : 1")
 
    fig, ax = plt.subplots(figsize=(6, 5))
    counts.plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])
    ax.set_title(f"Class Distribution - {label_col}")
    ax.set_xlabel(label_col)
    ax.set_ylabel("Count")
    savefig(fig, "class_distribution_bar.png")
 
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(counts.values, labels=[str(i) for i in counts.index],
           autopct="%1.2f%%", startangle=90, colors=["#4C72B0", "#DD8452"])
    ax.set_title(f"Class Distribution - {label_col}")
    savefig(fig, "class_distribution_pie.png")
 
    return imbalance_ratio
 
 

In [9]:
# ==============================================================================
# STEP 8 - DATA CLEANING
# ==============================================================================
 
def clean_data(df, label_col):
    """
    Clean the dataset in preparation for modeling:
      - drop obvious ID columns (kept for reference, not used as features)
      - drop duplicate rows (safety net; already done in Step 4)
      - handle any remaining missing values
      - convert categorical columns to numeric
      - ensure everything feeding the model is numeric
 
    Returns
    -------
    df_clean : pd.DataFrame  (cleaned, ML-ready, still includes label_col)
    id_cols  : list of columns identified and dropped as identifiers
    """
    print("\n" + "=" * 80)
    print("STEP 8: DATA CLEANING")
    print("=" * 80)
 
    df_clean = df.copy()
 
    # --- Identify and drop ID-like columns (but never the label column) ---
    id_cols = []
    for col in df_clean.columns:
        if col == label_col:
            continue
        low = col.lower()
        if any(kw == low or low.startswith(kw) for kw in ["id", "user_id", "userid"]):
            id_cols.append(col)
        elif low in ("employee_id", "employeeid"):  # Kept "user" for the user-level split later
            id_cols.append(col)
 
    if id_cols:
        print(f"Dropping identifier columns: {id_cols}")
        df_clean = df_clean.drop(columns=id_cols)
    else:
        print("No identifier columns detected to drop.")
 
    # --- Duplicate safety net ---
    n_before = len(df_clean)
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)
    print(f"Duplicate rows removed in cleaning stage: {n_before - len(df_clean)}")
 
    # --- Handle missing values ---
    missing_before = df_clean.isnull().sum().sum()
    if missing_before > 0:
        print(f"Handling {missing_before} missing values...")
        num_cols = df_clean.select_dtypes(include=[np.number]).columns
        cat_cols = df_clean.select_dtypes(exclude=[np.number]).columns
 
        for col in num_cols:
            if df_clean[col].isnull().any():
                median_val = df_clean[col].median()
                df_clean[col] = df_clean[col].fillna(median_val)
 
        for col in cat_cols:
            if df_clean[col].isnull().any():
                mode_val = df_clean[col].mode(dropna=True)
                fill_val = mode_val.iloc[0] if not mode_val.empty else "missing"
                df_clean[col] = df_clean[col].fillna(fill_val)
 
        print(f"[OK] Missing values after cleaning: {df_clean.isnull().sum().sum()}")
    else:
        print("No missing values to handle.")
 
    # --- Encode remaining categorical (non-numeric) columns ---
    cat_cols = df_clean.select_dtypes(exclude=[np.number]).columns.tolist()
    if label_col in cat_cols:
        cat_cols.remove(label_col)  # label handled separately below
 
    if cat_cols:
        print(f"Encoding categorical feature columns: {cat_cols}")
        for col in cat_cols:
            le = LabelEncoder()
            df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    else:
        print("No non-numeric feature columns require encoding.")
 
    # --- Ensure label column is numeric (0/1) ---
    if df_clean[label_col].dtype == object or df_clean[label_col].dtype.name == "category":
        print(f"Encoding label column '{label_col}' to numeric.")
        le = LabelEncoder()
        df_clean[label_col] = le.fit_transform(df_clean[label_col].astype(str))
 
    print(f"\n[OK] Cleaned dataset shape: {df_clean.shape}")
    print(f"[OK] All feature columns numeric: "
          f"{all(np.issubdtype(dt, np.number) for dt in df_clean.dtypes)}")
 
    return df_clean, id_cols
 

In [10]:

# ==============================================================================
# STEP 9 - FEATURE / TARGET PREPARATION
# ==============================================================================
 
def prepare_features(df_clean, label_col):
    """Split the cleaned dataframe into feature matrix X and target vector y."""
    print("\n" + "=" * 80)
    print("STEP 9: FEATURE PREPARATION")
    print("=" * 80)
 
    X = df_clean.drop(columns=[label_col])
    y = df_clean[label_col]
 
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
 
    return X, y

In [11]:
def split_data(X, y):
    """Perform an 80/20 user-level train/test split to prevent data leakage."""
    print("\n" + "=" * 80)
    print("STEP 10: TRAIN/TEST SPLIT (USER-LEVEL)")
    print("=" * 80)

    # 1. Identify unique users in the dataset
    # Note: 'user' might have been label encoded in step 8, but it still represents unique users.
    if 'user' not in X.columns:
        # Fallback if the column is named differently or was dropped
        print("[WARN] 'user' column not found. Falling back to random split.")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
        )
        return X_train, X_test, y_train, y_test

    users = X['user'].unique()
    
    # 2. Split the users 80/20
    from sklearn.model_selection import train_test_split
    train_users, test_users = train_test_split(
        users, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    
    # 3. Create boolean masks for train and test sets
    train_mask = X['user'].isin(train_users)
    test_mask = X['user'].isin(test_users)
    
    # 4. Filter X and y based on the masks
    X_train = X[train_mask].copy()
    X_test = X[test_mask].copy()
    y_train = y[train_mask].copy()
    y_test = y[test_mask].copy()
    
    # 5. Drop the 'user' column as it shouldn't be a feature for the ML model
    X_train = X_train.drop(columns=['user'])
    X_test = X_test.drop(columns=['user'])
    
    print(f"Training samples: {X_train.shape[0]} ({len(train_users)} users)")
    print(f"Testing samples : {X_test.shape[0]} ({len(test_users)} users)")
    
    # Verify no overlap
    overlap = set(train_users).intersection(set(test_users))
    print(f"User overlap between train/test: {len(overlap)}")

    return X_train, X_test, y_train, y_test


In [12]:
# ==============================================================================
# STEP 11 - CLASS IMBALANCE HANDLING (NO SYNTHETIC OVERSAMPLING)
# ==============================================================================
 
def report_class_balance(X_train, y_train):
    """
    Report the training-set class distribution and pass the training data
    through unchanged.
 
    This pipeline intentionally does NOT apply any synthetic oversampling
    (e.g. SMOTE). Class imbalance is instead handled downstream by the
    classifier's class_weight="balanced" setting (see train_random_forest),
    which reweights the loss during training rather than fabricating
    synthetic minority-class samples. X_train / y_train are therefore
    returned as-is.
    """
    print("\n" + "=" * 80)
    print("STEP 11: CLASS IMBALANCE HANDLING (NO SMOTE)")
    print("=" * 80)
 
    print("Training-set class counts (unchanged, no resampling applied):")
    print(y_train.value_counts())
    print(
        "\n[OK] No synthetic oversampling is applied. Class imbalance will be "
        "handled by the Random Forest's class_weight='balanced' setting."
    )
 
    X_train_res = X_train
    y_train_res = y_train
 
    return X_train_res, y_train_res
 

In [13]:
# ==============================================================================
# STEP 12 - TRAIN BASELINE MODEL
# ==============================================================================
 
def train_random_forest(X_train, y_train):
    """Train a baseline Random Forest classifier with reasonable defaults."""
    print("\n" + "=" * 80)
    print("STEP 12: TRAIN BASELINE MODEL (Random Forest)")
    print("=" * 80)
 
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight="balanced",   # reweights classes instead of oversampling
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
 
    print("Fitting Random Forest classifier...")
    model.fit(X_train, y_train)
    print("[OK] Model training complete.")
 
    return model
 
 

In [14]:
# ==============================================================================
# STEP 13 - MODEL EVALUATION
# ==============================================================================
 
def evaluate_model(model, X_test, y_test, feature_names):
    """
    Evaluate the trained model on the held-out test set and produce
    all required metrics and plots.
 
    Returns
    -------
    metrics_dict : dict of scalar metrics
    fi_df        : pd.DataFrame of feature importances
    """
    print("\n" + "=" * 80)
    print("STEP 13: MODEL EVALUATION")
    print("=" * 80)
 
    y_pred = model.predict(X_test)
    model_classes = list(model.classes_)
 
    # Full label set = every class either the model learned OR that appears
    # in the test set, sorted for stable, deterministic output ordering.
    all_labels = sorted(set(model_classes) | set(pd.Series(y_test).unique().tolist()))
    n_classes = len(all_labels)
    is_binary = n_classes == 2
 
    print(f"Classes detected in target: {all_labels}  "
          f"({'binary' if is_binary else 'multiclass'} classification)")
 
    # --- Predicted probabilities (needed for ROC-AUC / ROC curves) ---
    proba_matrix = model.predict_proba(X_test)  # columns follow model.classes_ order
 
    # --- Precision / Recall / F1 ---
    # 'binary' averaging only makes sense for a true 2-class problem; a
    # 4-class (or any >2-class) target must use a multiclass-aware average.
    # zero_division=0 avoids warnings/errors when a class has no predicted
    # or true samples in a given split.
    average_method = "binary" if is_binary else "weighted"
 
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average=average_method,
                            labels=all_labels if not is_binary else None,
                            zero_division=0)
    rec = recall_score(y_test, y_pred, average=average_method,
                        labels=all_labels if not is_binary else None,
                        zero_division=0)
    f1 = f1_score(y_test, y_pred, average=average_method,
                  labels=all_labels if not is_binary else None,
                  zero_division=0)
 
    # --- ROC-AUC ---
    # Binary: AUC of the positive (max-valued, typically 1) class.
    # Multiclass: one-vs-rest AUC averaged across classes ('weighted'
    # accounts for class imbalance across the 4 categories).
    # Both require every class used in the AUC calculation to actually be
    # present in y_test, and the model must have seen it during training.
    roc_auc = float("nan")
    try:
        if is_binary:
            if 1 in model_classes and proba_matrix.shape[1] == 2:
                positive_idx = model_classes.index(1)
                y_proba_binary = proba_matrix[:, positive_idx]
                if pd.Series(y_test).nunique() == 2:
                    roc_auc = roc_auc_score(y_test, y_proba_binary)
                else:
                    print("[WARN] Test set has only one class; ROC-AUC is undefined.")
            else:
                print(
                    f"[WARN] Model only learned class(es) {model_classes} during "
                    "training; ROC-AUC and the ROC curve will be skipped."
                )
        else:
            if set(pd.Series(y_test).unique().tolist()).issubset(set(model_classes)):
                roc_auc = roc_auc_score(
                    y_test, proba_matrix,
                    multi_class="ovr", average="weighted",
                    labels=model_classes,
                )
            else:
                print(
                    "[WARN] Test set contains a class the model never saw during "
                    "training; ROC-AUC will be skipped."
                )
    except ValueError as exc:
        print(f"[WARN] ROC-AUC could not be computed: {exc}")
 
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision ({average_method}): {prec:.4f}")
    print(f"Recall ({average_method})   : {rec:.4f}")
    print(f"F1-score ({average_method}) : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}" if not np.isnan(roc_auc) else "ROC-AUC   : N/A")
 
    # Explicit labels=all_labels keep the confusion matrix / report at a
    # fixed, consistent shape even if a class is briefly absent from a
    # given y_test or y_pred split.
    cm = confusion_matrix(y_test, y_pred, labels=all_labels)
    print("\nConfusion Matrix:")
    print(cm)
 
    report = classification_report(y_test, y_pred, labels=all_labels, zero_division=0)
    print("\nClassification Report:")
    print(report)
 
    # --- Confusion matrix plot ---
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=all_labels, yticklabels=all_labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix")
    savefig(fig, "confusion_matrix.png")
 
    # --- ROC curve plot ---
    if not np.isnan(roc_auc):
        fig, ax = plt.subplots(figsize=(6.5, 5.5))
        if is_binary:
            positive_idx = model_classes.index(1)
            fpr, tpr, _ = roc_curve(y_test, proba_matrix[:, positive_idx])
            ax.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.4f})", color="darkorange")
        else:
            # One-vs-rest ROC curve per class, using label_binarize so each
            # class gets its own binary "this class vs. everything else" curve.
            from sklearn.preprocessing import label_binarize
            y_test_bin = label_binarize(y_test, classes=model_classes)
            for i, cls in enumerate(model_classes):
                if y_test_bin[:, i].sum() == 0:
                    continue  # class absent from this test split, skip its curve
                fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba_matrix[:, i])
                class_auc = roc_auc_score(y_test_bin[:, i], proba_matrix[:, i])
                ax.plot(fpr, tpr, label=f"Class {cls} (AUC = {class_auc:.4f})")
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("ROC Curve" if is_binary else "ROC Curves (One-vs-Rest)")
        ax.legend(loc="lower right", fontsize=8)
        savefig(fig, "roc_curve.png")
    else:
        print("[INFO] Skipping ROC curve plot (ROC-AUC unavailable).")
 
    # --- Feature importance plot ---
    importances = model.feature_importances_
    fi_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances,
    }).sort_values("importance", ascending=False).reset_index(drop=True)
 
    top_n = min(25, len(fi_df))
    fig, ax = plt.subplots(figsize=(8, max(4, 0.3 * top_n)))
    sns.barplot(data=fi_df.head(top_n), x="importance", y="feature",
                ax=ax, color="seagreen")
    ax.set_title(f"Top {top_n} Feature Importances (Random Forest)")
    savefig(fig, "feature_importance.png")
 
    metrics_dict = {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "roc_auc": roc_auc,
    }
 
    # Save the raw classification report text alongside metrics
    with open(os.path.join(DIRS["metrics"], "classification_report.txt"), "w") as f:
        f.write(report)
 
    return metrics_dict, fi_df
 
 


In [15]:
# ==============================================================================
# STEP 14 - SAVE OUTPUTS
# ==============================================================================
 
def save_outputs(df_clean, model, metrics_dict, fi_df):
    """Persist the cleaned dataset, trained model, metrics, and feature importances."""
    print("\n" + "=" * 80)
    print("STEP 14: SAVE OUTPUTS")
    print("=" * 80)
 
    # Cleaned dataset
    clean_data_path = os.path.join(DIRS["data"], "dayr4.2_cleaned.csv")
    df_clean.to_csv(clean_data_path, index=False)
    print(f"[OK] Cleaned dataset saved to: {clean_data_path}")
 
    # Trained model
    model_path = os.path.join(DIRS["models"], "random_forest_model.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    print(f"[OK] Model saved to: {model_path}")
 
    # Evaluation metrics
    metrics_path = os.path.join(DIRS["metrics"], "evaluation_metrics.csv")
    pd.DataFrame([metrics_dict]).to_csv(metrics_path, index=False)
    print(f"[OK] Evaluation metrics saved to: {metrics_path}")
 
    # Feature importance
    fi_path = os.path.join(DIRS["metrics"], "feature_importance.csv")
    fi_df.to_csv(fi_path, index=False)
    print(f"[OK] Feature importance saved to: {fi_path}")
 
    print(f"\n[OK] All plots saved under: {DIRS['plots']}")
 


In [16]:
# ==============================================================================
# MAIN PIPELINE
# ==============================================================================
 
def main():
    """Run the complete Week 1 pipeline end to end."""
    try:
        make_output_dirs()
 
        # STEP 1
        df = load_dataset(DATASET_PATH)
 
        # STEP 2
        inspect_dataset(df)
 
        # STEP 3
        missing_value_analysis(df)
 
        # STEP 4
        df = duplicate_analysis(df)
 
        # STEP 5
        exploratory_data_analysis(df)
 
        # STEP 6
        label_col = identify_label_column(df)
 
        # STEP 7
        class_imbalance_analysis(df, label_col)
 
        # STEP 8
        df_clean, id_cols = clean_data(df, label_col)
 
        # STEP 9
        X, y = prepare_features(df_clean, label_col)
 
        # STEP 10
        X_train, X_test, y_train, y_test = split_data(X, y)
 
        # STEP 11
        X_train_res, y_train_res = report_class_balance(X_train, y_train)
 
        # STEP 12
        model = train_random_forest(X_train_res, y_train_res)
 
        # STEP 13
        metrics_dict, fi_df = evaluate_model(model, X_test, y_test, X.columns.tolist())
 
        # STEP 14
        save_outputs(df_clean, model, metrics_dict, fi_df)
 
        print("\n" + "=" * 80)
        print("WEEK 1 PIPELINE COMPLETE")
        print("=" * 80)
        print(f"Label column used     : {label_col}")
        print(f"ID columns dropped    : {id_cols}")
        print(f"Final dataset shape   : {df_clean.shape}")
        print(f"Test-set Accuracy     : {metrics_dict['accuracy']:.4f}")
        print(f"Test-set ROC-AUC      : {metrics_dict['roc_auc']:.4f}")
        print(f"All outputs saved to  : '{OUTPUT_DIR}/'")
 
    except FileNotFoundError as exc:
        print(f"\n[ERROR] {exc}")
        sys.exit(1)
    except Exception as exc:
        print(f"\n[ERROR] Pipeline failed with an unexpected error: {exc}")
        traceback.print_exc()
        sys.exit(1)
 
 
if __name__ == "__main__":
    main()
 
 

[OK] Output directories ready under 'outputs/'

STEP 1: LOAD DATASET
[OK] Loaded 'C:\insider_threat_detection_project2\data\dayr4.2.csv' successfully.

STEP 2: INITIAL DATASET INSPECTION

Shape                : (330452, 508)
Number of rows       : 330452
Number of columns    : 508

Column names:
    1. starttime
    2. endtime
    3. user
    4. day
    5. week
    6. isweekday
    7. isweekend
    8. role
    9. b_unit
   10. f_unit
   11. dept
   12. team
   13. ITAdmin
   14. O
   15. C
   16. E
   17. A
   18. N
   19. n_allact
   20. allact_n-pc0
   21. allact_n-pc1
   22. allact_n-pc2
   23. allact_n-pc3
   24. n_workhourallact
   25. workhourallact_n-pc0
   26. workhourallact_n-pc1
   27. workhourallact_n-pc2
   28. workhourallact_n-pc3
   29. n_afterhourallact
   30. afterhourallact_n-pc0
   31. afterhourallact_n-pc1
   32. afterhourallact_n-pc2
   33. afterhourallact_n-pc3
   34. n_logon
   35. logon_n-pc0
   36. logon_n-pc1
   37. logon_n-pc2
   38. logon_n-pc3
   39. n_workh

Traceback (most recent call last):
  File "C:\Users\Irwa\AppData\Local\Temp\ipykernel_20740\963345504.py", line 47, in main
    metrics_dict, fi_df = evaluate_model(model, X_test, y_test, X.columns.tolist())
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Irwa\AppData\Local\Temp\ipykernel_20740\2915624148.py", line 143, in evaluate_model
    fi_df = pd.DataFrame({
            ^^^^^^^^^^^^^^
  File "C:\insider_threat_detection_project2\.venv\Lib\site-packages\pandas\core\frame.py", line 782, in __init__
    mgr = dict_to_mgr(data, index, columns, dtype=dtype, copy=copy, typ=manager)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\insider_threat_detection_project2\.venv\Lib\site-packages\pandas\core\internals\construction.py", line 503, in dict_to_mgr
    return arrays_to_mgr(arrays, columns, index, dtype=dtype, typ=typ, consolidate=copy)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

SystemExit: 1